# Step 4: Embeddings
**AI Agents From Scratch — `ai-agents-from-scratch`**

Goals:
- Convert text into embedding vectors
- Measure semantic similarity between sentences
- Run a mini similarity search (foundation of RAG retrieval)

Model: `all-MiniLM-L6-v2` via `sentence-transformers` (free, runs locally)

In [1]:
# Install once
!pip install sentence-transformers -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load the embedding model

In [2]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

c:\Users\zaras\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\zaras\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 2. What does an embedding look like?

An embedding is just a list of numbers. Each sentence becomes a fixed-size vector regardless of how long the sentence is.

In [ ]:
sample = "Artificial intelligence is transforming software engineering."
embedding = model.encode(sample)

print(f"Text   : {sample}")
print(f"Vector shape : {embedding.shape}")
print(f"First 8 values: {embedding[:8].round(4)}")

## 3. Cosine similarity between two sentences

Cosine similarity ranges from **-1 to 1**. Closer to 1 = more similar meaning.

In [ ]:
pairs = [
    ("I love programming in Python.",     "Python is my favourite language."),   # should be HIGH
    ("I love programming in Python.",     "The weather is nice today."),          # should be LOW
    ("Machine learning models need data.","AI systems require large datasets."),  # should be HIGH
    ("Machine learning models need data.","I enjoy playing football."),            # should be LOW
]

for a, b in pairs:
    emb_a = model.encode(a)
    emb_b = model.encode(b)
    score = util.cos_sim(emb_a, emb_b).item()
    print(f"Score: {score:.4f}")
    print(f"  A: {a}")
    print(f"  B: {b}")
    print()

## 4. Similarity Search (Mini RAG Retrieval)

This is the exact mechanism used in RAG:
- Embed a collection of documents at index time
- Embed a query at runtime
- Find the closest documents by cosine similarity

In [ ]:
# Simulated knowledge base (could be chunks from any document)
documents = [
    "Transformers use attention mechanisms to process sequences.",
    "Python is widely used in data science and machine learning.",
    "RAG combines retrieval with language model generation.",
    "Neural networks learn by adjusting weights through backpropagation.",
    "Vector databases store embeddings for fast similarity search.",
    "Pakistan won the Cricket World Cup in 1992.",
    "Docker containers help with reproducible environments.",
    "LLMs are pre-trained on large corpora of text data.",
    "The Eiffel Tower is located in Paris, France.",
    "Embeddings convert text into numerical vectors that capture meaning.",
]

# Embed all documents once (index time)
doc_embeddings = model.encode(documents, convert_to_tensor=True)
print(f"Indexed {len(documents)} documents. Shape: {doc_embeddings.shape}")

In [ ]:
def similarity_search(query, top_k=3):
    """Find the top_k most relevant documents for a given query."""
    query_embedding = model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, doc_embeddings)[0]
    
    top_results = scores.topk(top_k)
    
    print(f"Query: \"{query}\"\n")
    for score, idx in zip(top_results.values, top_results.indices):
        print(f"  [{score:.4f}] {documents[idx]}")
    print()

In [ ]:
similarity_search("How do language models get trained?")
similarity_search("What is used to store vectors efficiently?")
similarity_search("How does retrieval augmented generation work?")

## 5. The RAG Pipeline — Visualised

```
[Documents]  -->  embed()  -->  [Vector Store]
                                      |
[User Query] -->  embed()  -->  cosine_similarity()
                                      |
                             [Top-K Relevant Chunks]
                                      |
                         prompt = query + chunks
                                      |
                              [LLM generates answer]
```

The `similarity_search()` function above **is** the retrieval step of RAG.

In a real system, the vector store would be something like **ChromaDB**, **FAISS**, or **Pinecone** — but the logic is identical.

## 6. Try your own query

In [ ]:
your_query = "explain attention in transformers"  # change this
similarity_search(your_query, top_k=3)

---
## Summary

| Concept | What you did |
|---|---|
| Embedding | Converted sentences → 384-dim vectors |
| Cosine similarity | Measured semantic closeness between pairs |
| Similarity search | Retrieved top-K relevant docs for a query |
| RAG connection | The search step feeds context into an LLM |

**Next:** Step 5 — Vector Databases (ChromaDB), where you persist these embeddings and do this at scale.